# KjarniEmbeddings

This will help you get started with [Kjarni](https://kjarni.ai) embedding models using LangChain.

Kjarni is a native inference engine that runs embedding models in your own process. It produces the same vectors as `sentence-transformers` for the same model, without PyTorch, ONNX Runtime, a GPU or a server, and needs no API key. The model downloads once on first use and is cached; after that everything works offline.

## Overview
### Integration details

| Provider | Package |
|:--------:|:-------:|
| [Kjarni](https://kjarni.ai) | [langchain-kjarni](https://pypi.org/project/langchain-kjarni/) |

## Setup

There are no credentials to set up. Models are downloaded from Hugging Face on first use and cached in Kjarni's cache directory (`~/.cache/kjarni` on Linux and macOS).

### Installation

The LangChain Kjarni integration lives in the `langchain-kjarni` package. It depends only on `langchain-core` and `kjarni`, which has no Python dependencies of its own.

In [1]:
%pip install -qU langchain-kjarni

Note: you may need to restart the kernel to use updated packages.


## Instantiation

Now we can instantiate our model object and generate embeddings. Construction is free: the model loads on first use.

In [2]:
from langchain_kjarni import KjarniEmbeddings

embeddings = KjarniEmbeddings(model="minilm-l6-v2")

## Indexing and Retrieval

Embedding models are often used in retrieval-augmented generation (RAG) flows, both as part of indexing data as well as later retrieving it. For more detailed instructions, please see our [RAG tutorials](https://python.langchain.com/docs/tutorials/rag/).

Below, see how to index and retrieve data using the `embeddings` object we initialized above. In this example, we will index and retrieve a sample document in the `InMemoryVectorStore`.

In [3]:
# Create a vector store with a sample text
from langchain_core.vectorstores import InMemoryVectorStore

text = "LangChain is the framework for building context-aware reasoning applications"

vectorstore = InMemoryVectorStore.from_texts(
    [text],
    embedding=embeddings,
)

# Use the vectorstore as a retriever
retriever = vectorstore.as_retriever()

# Retrieve the most similar text
retrieved_documents = retriever.invoke("What is LangChain?")

# show the retrieved document's content
retrieved_documents[0].page_content

'LangChain is the framework for building context-aware reasoning applications'

## Direct Usage

Under the hood, the vectorstore and retriever implementations are calling `embeddings.embed_documents(...)` and `embeddings.embed_query(...)` to create embeddings for the text(s) used in `from_texts` and retrieval `invoke` operations, respectively.

You can directly call these methods to get embeddings for your own use cases.

### Embed single texts

You can embed single texts or documents with `embed_query`:

In [4]:
single_vector = embeddings.embed_query(text)
print(str(single_vector)[:100])  # Show the first 100 characters of the vector

[-0.019566863775253296, -0.0075781699270009995, -0.013376356102526188, -0.0650729313492775, 0.033896


### Embed multiple texts

You can embed multiple texts with `embed_documents`:

In [5]:
text2 = (
    "LangGraph is a library for building stateful, multi-actor applications with LLMs"
)
two_vectors = embeddings.embed_documents([text, text2])
for vector in two_vectors:
    print(str(vector)[:100])  # Show the first 100 characters of the vector

[-0.019566895440220833, -0.0075781517662107944, -0.013376353308558464, -0.06507297605276108, 0.03389
[-0.02836061269044876, -0.07940301299095154, -0.01885548233985901, -0.09558400511741638, -0.03100424


## Choosing a model

| `model` | Dimensions | Notes |
|:--|:--:|:--|
| `minilm-l6-v2` (default) | 384 | all-MiniLM-L6-v2. Fast, English. |
| `mpnet-base-v2` | 768 | all-mpnet-base-v2. Higher quality, English. |
| `nomic-embed-text` | 768 | Long inputs (up to 8192 tokens). |
| `bge-m3` | 1024 | Multilingual. |

`kjarni model list` (from the [Kjarni CLI](https://kjarni.ai/docs/)) shows every available model. For models trained with instruction prefixes, set `query_prefix` and `document_prefix`. Set `device="gpu"` to run on the GPU through Vulkan, Metal or DirectX 12.

## API Reference

For detailed documentation of all `KjarniEmbeddings` features and configurations, see the [langchain-kjarni README](https://github.com/olafurjohannsson/langchain-kjarni) and the [Kjarni documentation](https://kjarni.ai/docs/).